# Deliverable 4 - INF201 

Submission deadline: 27th Nov 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit Python scripts (.py), a Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebook
    * See Overview and objectives for a list of what your notebook should contain
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together
* We would appriciate it if use English for this assignment, both in the writing and in the video

**Group members:** Mahdi Heydari and Noh 

**GitHub repository:** https://github.com/Mehdihey/inf201-deliverable4

Files in the repository:
* `temperature.py`, `test_temperature.py` (Task 1)
* `fraction.py`, `test_fraction.py` (Task 2)
* `requirements.txt`, `git_log.txt`, this notebook and its PDF

## Task 1: Temperatures

Create a `Temperature` class that supports Celsius ("C"), Fahrenheit ("F"), and Kelvin ("K"). This class should:
1. Be initialised with a value and a unit (case-insensitive), and raise a `ValueError` if the unit is not one of "C", "F" or "K", or if the temperature is below absolute zero (0 K).
2. Use a tolerance of 1e-9 for floating-point value comparisons
3. Expose `value` and `unit` as read-only attributes
4. Contain the following conversion methods:
    * `to_celsius(self) -> float`:
        * From Kelvin: $C = K - 273.15$
        * From Fahrenheit: $C = (F - 32) \times \frac{5}{9}$
    * `to_fahrenheit(self) -> float`:
        * From Celsius: $F = C \times \frac{9}{5} + 32$
        * From Kelvin: $F = (K - 273.15) \times \frac{9}{5} + 32$
    * `to_kelvin(self) -> float`:
        * From Celsius: $K = C + 273.15$
        * From Fahrenheit: $K = (F - 32) \times \frac{5}{9} + 273.15$
    * `convert_to(self, target_unit: str) -> Temperature`:
        * Returns a new `Temperature` instance converted to `target_unit` (case-insensitive), raising `ValueError` if `target_unit` is invalid.
5. Contain the following string representations:
    * `__repr__(self) -> str`: Returns an unambiguous representation, such as `Temperature(25.0, 'C')`.
    * `__str__(self) -> str`: Returns a human-readable string formatted to two decimal places, such as `25.00 °C`.
6. Contain all six comparisons (==, !=, <, <=, >, >=) so temperatures can be compared across different units (`Temperature(0, "C") == Temperature(32, "F")`). Should return `NotImplemented` if a `Temperature` instance is compared to a non-Temperature instance.

Write a `test_temperature.py` script that imports the `Temperature` class and uses pytest for the following tests:
1. Initialization: Default unit, case-insensitive units, and read-only behavior.
2. Validation: Invalid unit strings and temperatures below absolute zero in "C", "F", and "K".
3. Conversions: Test conversion accuracy across reference points:
    * Absolute zero ($-273.15^\circ\text{C} = -459.67^\circ\text{F} = 0\text{ K}$)
    * Celsius–Fahrenheit crossover ($-40^\circ\text{C} = -40^\circ\text{F} = 233.15\text{ K}$)
    * Freezing point of water ($0^\circ\text{C} = 32^\circ\text{F} = 273.15\text{ K}$)
    * Human body temperature ($37^\circ\text{C} = 98.6^\circ\text{F} = 310.15\text{ K}$)
    * Boiling point of water ($100^\circ\text{C} = 212^\circ\text{F} = 373.15\text{ K}$)
4. Round-trip conversions: Converting "C" -> "F" -> "K" -> "C" reproduces the original Temperature.
5. Cross-unit comparisons: Equality and ordering (<, <=, >, >=) across mixed units.
6. Formatting: Outputs of `str()` and `repr()`

### Solution to Task 1

The class is in `temperature.py`. Design choices:
* Every temperature is converted to **Kelvin** internally, both for the absolute-zero check and for all comparisons, so mixed units can be compared directly.
* `value` and `unit` are stored as `_value` and `_unit` and exposed through `@property` without a setter, which makes them read-only.
* Equality uses a tolerance of `1e-9`, because floating-point arithmetic is not exact (e.g. -459.67 °F converts to about -6e-14 K, not exactly 0 K).
* Comparisons with non-`Temperature` objects return `NotImplemented`, so Python itself returns `False` for `==` and raises `TypeError` for `<`.

In [ ]:
from temperature import Temperature

t = Temperature(37, "c")
print(repr(t))
print(t)
print(t.convert_to("F"), "|", t.convert_to("K"))
print(Temperature(0, "C") == Temperature(32, "F"))
print(Temperature(0, "C") < Temperature(300, "K"))

#### Test results for Task 1

The tests in `test_temperature.py` are run with pytest:

In [ ]:
import sys
!{sys.executable} -m pytest -v test_temperature.py

## Task 2: Fractions

Create a `Fraction` class representing fractions. Both numerator and denominator should be integers, and the denominator should be stricktly positive (>0). The class should implement the following special methods (aka magic methods, dunder methods): 
* initialiser
* all comparisons (equal, not equal, smaller than, smaller than or equal, greater than, and greated than or equal)
* at least four other special methods

Write a test script to be used with `pytest` for testing the implemented methods. Include the output from testing in a Jupyter notebok.

### Solution to Task 2

The class is in `fraction.py`. Design choices:
* The initialiser checks that numerator and denominator are integers (`TypeError` otherwise, `bool` is rejected) and raises `ZeroDivisionError` for a zero denominator.
* A negative denominator moves its sign to the numerator (`1/-2` becomes `-1/2`), so the denominator is always strictly positive.
* Fractions are reduced to lowest terms with `math.gcd` (`2/4` becomes `1/2`).
* Comparisons use cross-multiplication: `a/b < c/d` exactly when `a*d < c*b`. This is only valid because both denominators are positive.
* Besides `__init__` and the six comparisons, the class implements eight more special methods: `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__neg__`, `__float__`, `__repr__` and `__str__`.

In [ ]:
from fraction import Fraction

a = Fraction(1, 2)
b = Fraction(2, -6)
print(repr(b), "->", b)
print(f"{a} + {b} = {a + b}")
print(f"{a} - {b} = {a - b}")
print(f"{a} * {b} = {a * b}")
print(f"{a} / {b} = {a / b}")
print(f"-{a} = {-a},  float({a}) = {float(a)}")
print(a == Fraction(2, 4), a > b, a <= b)

#### Test results for Task 2

The tests in `test_fraction.py` are run with pytest:

In [ ]:
!{sys.executable} -m pytest -v test_fraction.py

## Use of generative AI

We used Claude (Anthropic) as a tutor and coding assistant in this deliverable.

**What we used it for**
* Explaining the assignment text in simple terms before we started.
* Getting a first version of the `Temperature` and `Fraction` classes and their pytest tests, which we then went through line by line.
* Setting up the virtual environment, `requirements.txt`, `.gitignore` and the GitHub repository.
* Debugging: when we typed the code ourselves we got many errors. Instead of asking for fixed code, we asked the AI to explain how to read the pytest output so we could find the mistakes ourselves.

**Mistakes we found and fixed ourselves (with guidance)**
* Wrong file names (`tempratur.py`, `Test.py`) so `import` and pytest could not find the files.
* Missing commas and parentheses in `@pytest.mark.parametrize` lists.
* Wrong formula (`* 9` instead of `* 5 / 9`) and a stray `._` in `self._unit._`.
* `__repr__` and `__str__` placed inside `__init__`, which made them local functions instead of methods.
* Typos in expected strings in the tests (e.g. missing `°`, missing two decimals).

**What we learned**
* How to read pytest output: the `>` line shows where the error is and the `E` lines show expected vs. actual value.
* That indentation decides which function or class a line belongs to.
* That a failing test can mean the test is wrong, not only the code.

**Our assessment:** The AI was most useful as an explainer and for debugging strategy. We checked all code by running the tests ourselves, and we are able to explain every part of it.

## Git log

Generated after the last commit with:

`git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`

The file `git_log.txt` is included in the submission.